# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Question shape:** yes/no with an observed label — `is_declining_label` (1 when `trend_direction == "down"`, ~54% of rows). Per the toolkit, that shape starts with **Logistic Regression** (readable, gives coefficients I can sanity-check), then **Random Forest** (captures interactions my hand-built W04 rule had to *guess* at manually — the rule hard-codes "stale AND CTR-gap" as one fixed AND condition; a tree model can find that kind of interaction, and others I didn't think to encode, on its own).

I'm skipping Gradient Boosting and clustering here: this is a supervised yes/no task, not an unsupervised "group these" or open-ended "what drives X" question, so clustering doesn't fit. A few points of AUC from boosting isn't worth losing the readable coefficients and split-level feature importances I get from the two simpler models — simplicity is a feature at this stage, not a shortcut.

**Banned as inputs** (confirmed against `docs/data-dictionary.md` and my own W04 leakage check): `trend_direction` / `trend_pct` (the label's source) and `is_declining_label` itself (the target).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# reuse the repo's own precision@k implementation instead of writing a second copy
sys.path.append(str(Path("../../scripts").resolve()))
from ml_utils import precision_at_k

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
csv_path = next(p for p in candidates if p.exists())
df = pd.read_csv(csv_path)

# same label definition as docs/data-dictionary.md: is_declining_label = (trend_direction == "down")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"{len(df):,} rows, {df['client_id'].nunique()} clients, "
      f"declining rate: {df['is_declining_label'].mean():.3f}")

30,000 rows, 32 clients, declining rate: 0.542


## 2. Split design

**Grouped by client, not a random row split.** `client_id` is a pseudonym for grouping/joins only, per the data dictionary — never a feature. But a random row split would let the model see some of a client's pages in training and then recognize that *same client's* other pages in the test set, through correlated traits (a client's typical `content_type` mix, general traffic level, etc.) rather than learning something that actually generalizes across clients.

32 distinct clients. I'm holding out ~20% of **clients** entirely (not rows) for testing — the same idea as `scripts/03_train_model.py`'s `client_holdout`, reproduced directly here so the split logic is visible in this notebook without needing to open that file. Fixed seed (42), so this is reproducible.

In [2]:
client_ids = df["client_id"].unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(client_ids)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])

test_mask = df["client_id"].isin(test_clients)
train_df = df[~test_mask].reset_index(drop=True)
test_df = df[test_mask].reset_index(drop=True)

print(f"train: {len(train_df):,} rows / {train_df['client_id'].nunique()} clients")
print(f"test:  {len(test_df):,} rows / {test_df['client_id'].nunique()} clients")
print(f"train declining rate: {train_df['is_declining_label'].mean():.3f}")
print(f"test declining rate:  {test_df['is_declining_label'].mean():.3f}")

assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"])), \
    "client leakage across the split!"

train: 27,675 rows / 26 clients
test:  2,325 rows / 6 clients
train declining rate: 0.555
test declining rate:  0.391


## 3. Train + compare vs my baseline

**My W04 rule, recomputed here on this split** — same logic as `w04_baseline_score.ipynb`: a page is `stale_and_ctr_underperform` when `days_since_last_update >= 91` AND its `ctr` is under half its `position_tier`'s weighted CTR benchmark, scored by `impressions_90d`.

**One honest change from the original W04 version:** there, the CTR benchmark was computed over the *whole* dataset. Here I compute it from **train-client rows only**, then apply it to the test rows — otherwise the baseline would be peeking at test-set statistics that the model never gets to see, which would make the comparison unfair in the baseline's favor.

**Metric: precision@K** (K=20, K=50), same metric family as the `training-honest-models` skill and the repo's own `ml_utils.precision_at_k`. This is a ranking/queue problem, not a threshold classification problem, so precision@K — "how good is the top of the queue" — is the metric that actually matches the decision editors make.

In [3]:
# --- Recompute my W04 rule, fit only on train, applied to test ---
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

train_tier_ctr = (
    train_df[train_df["position_tier"].isin(position_order)]
    .groupby("position_tier")
    .apply(lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100)
)

def score_baseline(frame):
    tier_benchmark = frame["position_tier"].map(train_tier_ctr)
    stale = (frame["days_since_last_update"] >= 91).astype(int)
    ctr_gap = (
        (frame["ctr"] < 0.5 * tier_benchmark) & frame["position_tier"].isin(position_order)
    ).astype(int)
    return stale * ctr_gap * frame["impressions_90d"]

test_df = test_df.copy()
test_df["baseline_score"] = score_baseline(test_df).fillna(0)

baseline_p20 = precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 20)
baseline_p50 = precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 50)
print(f"W04 rule on test clients -- precision@20: {baseline_p20:.3f}, precision@50: {baseline_p50:.3f}")

# --- Features: safe list only, no label-source or product-flag columns ---
NUMERIC_FEATURES = [
    "days_since_last_update", "ctr", "avg_position", "impressions_90d", "clicks_90d",
    "engagement_rate", "scroll_rate", "content_age_days", "word_count",
]
CATEGORICAL_FEATURES = ["position_tier", "freshness_tier", "content_type", "main_intent"]

def build_features(frame):
    numeric = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
    categorical = pd.get_dummies(
        frame[CATEGORICAL_FEATURES].fillna("unknown").astype(str),
        prefix=CATEGORICAL_FEATURES,
    )
    return pd.concat([numeric.reset_index(drop=True), categorical.reset_index(drop=True)], axis=1)

X_train = build_features(train_df)
y_train = train_df["is_declining_label"]
X_test = build_features(test_df).reindex(columns=X_train.columns, fill_value=0)
y_test = test_df["is_declining_label"]

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

results = {
    "W04_baseline_rule": {"precision_at_20": baseline_p20, "precision_at_50": baseline_p50, "roc_auc": None},
}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "precision_at_20": precision_at_k(y_test, proba, 20),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "roc_auc": roc_auc_score(y_test, proba),
    }

base_rate = y_test.mean()
results["base_rate_random_ranking"] = {
    "precision_at_20": base_rate, "precision_at_50": base_rate, "roc_auc": 0.5,
}

comparison_table = pd.DataFrame(results).T.round(3)
comparison_table

W04 rule on test clients -- precision@20: 0.300, precision@50: 0.180


/opt/homebrew/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,precision_at_20,precision_at_50,roc_auc
W04_baseline_rule,0.300,0.180,NaN
logistic_regression,0.300,0.580,0.691
random_forest,0.600,0.600,0.735
base_rate_random_ranking,0.391,0.391,0.500


## 4. Errors and interpretation

- **Did any model beat the W04 rule?** Yes, clearly. Random Forest scores precision@20 = 0.600 and precision@50 = 0.600 -- well above the W04 rule's 0.300 / 0.180, and above the 0.391 base rate from random ranking. Logistic Regression ties the rule at precision@20 (0.300) but pulls ahead at precision@50 (0.580). Worth flagging plainly: **my W04 rule actually falls below random ranking at precision@50** (0.180 vs. 0.391 base rate) on this held-out set of clients -- the hand-built stale-AND-CTR-gap condition doesn't generalize well to clients it wasn't built against. Keeping that result rather than hiding it, same as the W02 finding.
- **What does the winning model lean on?** Random Forest's top features are `impressions_90d` (0.24), `content_age_days` (0.16), and `avg_position` (0.16) -- no single feature dominates (nothing near 0.9+), which is a reasonable sign against leakage. All three make sense: high-traffic, older, well-ranked pages are exactly where a real decline is both easier to measure and more likely to be a genuine trend rather than noise.

Read below before believing the score.

In [4]:
best_name = max(("logistic_regression", "random_forest"),
                 key=lambda n: results[n]["precision_at_50"])
best_model = fitted[best_name]
print(f"Best model by precision@50: {best_name}")
print()

# --- Feature importance / coefficients, to sanity-check what the model leans on ---
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(best_model.feature_importances_, index=X_train.columns)
else:
    importance = pd.Series(best_model.coef_[0], index=X_train.columns).abs()
importance = importance.sort_values(ascending=False)
print("Top 10 features:")
print(importance.head(10))
print()

# --- Where is it most wrong? ---
test_df["predicted_proba"] = best_model.predict_proba(X_test)[:, 1]
test_df["predicted_label"] = (test_df["predicted_proba"] >= 0.5).astype(int)
test_df["error_type"] = np.select(
    [
        (test_df["is_declining_label"] == 1) & (test_df["predicted_label"] == 0),
        (test_df["is_declining_label"] == 0) & (test_df["predicted_label"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)
print(test_df["error_type"].value_counts())
print()
print("Error rate by position_tier:")
print(test_df.groupby("position_tier")["error_type"].apply(lambda s: (s != "correct").mean()).round(3))
print()

# --- 3 concrete wrong cases, ranked by how confidently the model got them wrong ---
wrong = test_df[test_df["error_type"] != "correct"].copy()
wrong["confidence_gap"] = (wrong["predicted_proba"] - 0.5).abs()
wrong = wrong.sort_values("confidence_gap", ascending=False)

cols_to_show = [
    "content_id", "position_tier", "ctr", "days_since_last_update",
    "impressions_90d", "trend_direction", "predicted_proba", "error_type",
]
wrong[cols_to_show].head(3)

# TODO after running: add a markdown cell below with one sentence per case explaining
# *why* each one is hard -- e.g. a page with strong signals pointing one way but the
# opposite actual outcome. Same spirit as the W04 top-10 "what would make it wrong" review.

Best model by precision@50: random_forest

Top 10 features:
impressions_90d           0.237914
content_age_days          0.159052
avg_position              0.158199
word_count                0.080422
clicks_90d                0.060721
ctr                       0.054978
scroll_rate               0.054037
days_since_last_update    0.044704
position_tier_top_3       0.034990
engagement_rate           0.019909
dtype: float64

error_type
correct           1544
false_positive     537
false_negative     244
Name: count, dtype: int64

Error rate by position_tier:
position_tier
deep        0.417
page_1      0.391
page_3_5    0.448
striking    0.415
top_3       0.091
Name: error_type, dtype: float64



,content_id,position_tier,ctr,days_since_last_update,impressions_90d,trend_direction,predicted_proba,error_type
456,content_28b4223f4e5f,top_3,0.0,1,1,down,0.068566,false_negative
306,content_34b14c00f80c,top_3,0.0,20,3,down,0.073404,false_negative
2094,content_79ac977c6e0b,top_3,0.0,8,3,down,0.165163,false_negative


## Error read

The model is far more reliable in the `top_3` position tier (9.1% error rate) than anywhere else (39-45% error rate in `page_1`, `page_3_5`, `striking`, `deep`) -- right now it's really only trustworthy for the best-ranked pages.

All three of the most confidently-wrong cases are **false negatives on near-zero-traffic `top_3` pages**: 1, 3, and 3 total impressions over 90 days, each at 0% CTR, updated very recently (1-20 days ago). These are exactly the near-zero-base pages that made `trend_pct` unreliable in the first place (flagged back in W01/W02) -- a page swinging from "1 click" to "0 clicks" registers as `trend_direction == down`, but there's no real signal in 1-3 impressions to learn from. The model treats these as low-risk because everything else about the page looks fine; the label itself is the noisy part here, not the model.

**Honest verdict:** Random Forest beats both the W04 rule and random ranking, and what it leans on feels intuitive rather than suspicious. But its real strength is narrow -- solidly reliable only for `top_3` pages so far. Before trusting this for the capstone's ranked queue, the label needs the same near-zero-traffic floor I'd apply to any baseline (e.g. only score `is_declining_label` as meaningful above some minimum impression count), and error rates outside `top_3` need to come down before this is production-ready.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.